# RunPod H100：Qwen3 Steam 实体链接训练

此 Notebook 是云端执行入口，不复制训练实现。请先把它复制到仓库外的 `/workspace/runpod_training.ipynb`，再从上到下逐格执行：环境准备 → 32 条冒烟训练 → 完整训练 → 五个 checkpoint 评测 → 人工发布。这样 Jupyter 保存执行输出时不会把 Git 仓库变脏。

目标环境：1×H100 SXM 80GB、125GB RAM、16 vCPU、40GB 磁盘、`runpod/pytorch:1.0.2-cu1281-torch280-ubuntu2404`。实际训练和评测均由 `poc_a/scripts/*.py` 完成。

In [ ]:
from __future__ import annotations

import getpass
import json
import os
import subprocess
import sys
from pathlib import Path

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path('/workspace/qwen-steam-entity-linking')]
PROJECT_DIR = next((path.resolve() for path in candidates if (path / 'poc_a/configs/qwen3_8b_lora.yaml').is_file()), None)
if PROJECT_DIR is None:
    raise RuntimeError('未找到项目。请先把 GitHub 仓库克隆到 /workspace，再打开此 Notebook。')

POC_DIR = PROJECT_DIR / 'poc_a'
CONFIG = POC_DIR / 'configs/qwen3_8b_lora.yaml'
SMOKE_RUN_DIR = POC_DIR / 'outputs/runpod-smoke'
FULL_RUN_DIR = POC_DIR / 'outputs/runpod-full'
os.environ.setdefault('HF_HOME', '/workspace/.cache/huggingface')

def run(command: list[str]) -> None:
    print('$', ' '.join(command), flush=True)
    process = subprocess.Popen(
        command,
        cwd=PROJECT_DIR,
        env=os.environ.copy(),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

def latest_resumable_checkpoint(run_dir: Path) -> Path:
    candidates = []
    for checkpoint in (run_dir / 'checkpoints').glob('checkpoint-*'):
        if (checkpoint / 'optimizer.pt').is_file() and (checkpoint / 'scheduler.pt').is_file():
            candidates.append(checkpoint)
    if not candidates:
        raise RuntimeError(f'{run_dir} 中没有可恢复 checkpoint')
    return max(candidates, key=lambda path: int(path.name.rsplit('-', 1)[1]))

print('PROJECT_DIR =', PROJECT_DIR)
print('POC_DIR =', POC_DIR)
print('HF_HOME =', os.environ['HF_HOME'])

## 1. 检查环境并安装云端依赖

安装命令不会重新安装 PyTorch；CUDA PyTorch 2.8 由 RunPod 镜像提供。另行安装 `hf_transfer`，避免 RunPod 已启用 Hugging Face 加速下载但镜像缺少对应模块。脚本训练前还会检查 GPU、VRAM、RAM、CPU、磁盘和版本。

In [ ]:
run(['nvidia-smi'])
run(['df', '-h', '/workspace'])
run(['git', 'status', '--short'])
run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-r', str(POC_DIR / 'requirements-cloud.txt')])
run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'hf_transfer>=0.1.9,<1'])

## 2. 可选：注入 Hugging Face Token

公开基础模型通常无需 token。发布公开 LoRA 时必须使用具有 write 权限的 token；输入内容只保存在当前 Kernel 环境变量，不写入 Notebook。

In [ ]:
hf_token = getpass.getpass('HF_TOKEN（暂不需要可直接回车）：').strip()
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print('HF_TOKEN 已注入当前 Kernel；不会显示或写入文件。')
else:
    print('未设置 HF_TOKEN。')

## 3. 运行 32 条冒烟训练

连续两个 epoch 达到 100% canonical next-token 准确率后自动停止，最多 100 epochs。若目录已经存在，请先确认是否需要保留其结果并修改 `SMOKE_RUN_DIR`。

In [ ]:
run([
    sys.executable, str(POC_DIR / 'scripts/train.py'),
    '--config', str(CONFIG),
    '--mode', 'smoke',
    '--run-dir', str(SMOKE_RUN_DIR),
])

## 4. 运行完整 4000 行训练

1000 个实体分别覆盖 4 种 prompt；在第 2、4、6、8、10 个 epoch 保存 LoRA。40GB 磁盘策略只让最新 checkpoint 保留 optimizer/scheduler/RNG 恢复状态。

In [ ]:
run([
    sys.executable, str(POC_DIR / 'scripts/train.py'),
    '--config', str(CONFIG),
    '--mode', 'full',
    '--run-dir', str(FULL_RUN_DIR),
])

### 仅在完整训练中断后执行：从最新 checkpoint 恢复

正常完成训练时跳过本格。

In [ ]:
RESUME_FULL_TRAINING = False
if RESUME_FULL_TRAINING:
    checkpoint = latest_resumable_checkpoint(FULL_RUN_DIR)
    print('恢复：', checkpoint)
    run([
        sys.executable, str(POC_DIR / 'scripts/train.py'),
        '--config', str(CONFIG),
        '--mode', 'full',
        '--resume-from', str(checkpoint),
    ])
else:
    print('已跳过恢复训练。')

## 5. 评测五个里程碑并选择发布版本

In [ ]:
run([
    sys.executable, str(POC_DIR / 'scripts/evaluate.py'),
    '--run-dir', str(FULL_RUN_DIR),
    '--all-milestones',
])

In [ ]:
metrics = json.loads((FULL_RUN_DIR / 'metrics.json').read_text(encoding='utf-8'))
print('acceptance_passed =', metrics['acceptance_passed'])
print('selection =', json.dumps(metrics['selection'], ensure_ascii=False, indent=2))
for item in metrics['checkpoints']:
    print(
        f"epoch={item['epoch']:>2}  "
        f"canonical_entity_top1={item['canonical']['entity_top1_accuracy']:.2%}  "
        f"alias_entity_top1={item['alias']['entity_top1_accuracy']:.2%}  "
        f"alias_entity_top5={item['alias']['entity_top5_accuracy']:.2%}"
    )

## 6. 人工发布公开 Hugging Face LoRA

PoC A 的仓库 ID 直接写在下一个代码单元中，并先执行 dry-run。如果需要丢弃目标仓库的全部旧内容和历史，再单独启用重置单元。最后一格默认关闭公开写入，只有将 `PUBLISH_PUBLIC` 改为 `True` 才会上传。

In [ ]:
HF_REPO_ID = 'hxgdzyuyi/qwen3-8b-steam-entity-linking'
print('PoC A model repository =', HF_REPO_ID)
run([
    sys.executable, str(POC_DIR / 'scripts/publish_hf.py'),
    '--run-dir', str(FULL_RUN_DIR),
    '--repo-id', HF_REPO_ID,
    '--public',
    '--dry-run',
])

### 可选：清空旧的远程仓库

仅当目标仓库中的全部旧内容和提交历史都不再需要时执行。删除 Hugging Face 仓库不可恢复；本地 `FULL_RUN_DIR` 不受影响。本格默认跳过，启用后还必须输入完整确认语句。下一格的发布脚本会重新创建同名 private Model Repository，验证通过后再改为 public。

In [ ]:
RESET_REMOTE_REPOSITORY = False
if RESET_REMOTE_REPOSITORY:
    token = os.environ.get('HF_TOKEN')
    if not token:
        raise RuntimeError('重置远程仓库需要通过上面的 Secret 单元注入 HF_TOKEN')

    from huggingface_hub import HfApi

    api = HfApi(token=token)
    identity = api.whoami()
    print('当前 Hugging Face 身份 =', identity.get('name', '<unknown>'))
    print('即将永久删除 Model Repository =', HF_REPO_ID)
    print('当前远程文件：')
    for filename in api.list_repo_files(repo_id=HF_REPO_ID, repo_type='model'):
        print(' ', filename)

    required_confirmation = f'DELETE {HF_REPO_ID}'
    confirmation = input(f'\n确认永久删除请输入：{required_confirmation}\n> ').strip()
    if confirmation != required_confirmation:
        raise RuntimeError('输入不匹配，已取消删除')

    api.delete_repo(
        repo_id=HF_REPO_ID,
        repo_type='model',
        missing_ok=False,
    )
    print('远程仓库已删除。现在可以执行下一格公开发布。')
else:
    print('已跳过远程仓库重置。')

In [ ]:
PUBLISH_PUBLIC = False
if PUBLISH_PUBLIC:
    if not os.environ.get('HF_TOKEN'):
        raise RuntimeError('公开发布需要通过上面的 Secret 单元注入 HF_TOKEN')
    run([
        sys.executable, str(POC_DIR / 'scripts/publish_hf.py'),
        '--run-dir', str(FULL_RUN_DIR),
        '--repo-id', HF_REPO_ID,
        '--public',
    ])
else:
    print('公开上传默认关闭。确认指标和目标仓库后，将 PUBLISH_PUBLIC 改为 True。')